# Milestone 1 Lab, notebook 4 of 6: What is a tool, what is an agent

## Where we are

| Notebook | What you do |
|---|---|
| 1. The business question | Read the brief, explore the data, and answer the question by hand |
| 2. Your first LLM calls | Call a model, get a structured reply, and have AI draft the metric definitions |
| 3. Ask the LLM with all the data | Put the data in the prompt, ask the question, and check the answer |
| **4. What is a tool, what is an agent** (you are here) | Build one tool, run the agent loop by hand, then let LangChain run it |
| 5. The agent answers the question | Give an agent six tools and compare it with the single call |
| 6. Verify and go to production | Check an answer with code, and see the production version |

Model requests in this notebook: **about 7**. OpenRouter's free models allow about 50 a day.

## What this notebook is for

In notebook 3 the model had to do arithmetic, and we had no way to see where its numbers came from.

The fix is an idea called a **tool**. This notebook builds one from scratch, slowly, so that nothing is magic:

1. Write a tool. It's an ordinary Python function.
2. Tell the model the tool exists.
3. Watch the model ask us to run it.
4. Run it and send the result back.
5. Then see how **LangChain** does steps 2 to 4 for us.

By the end you will know what an agent is and why we use LangChain to build one.

In [ ]:
import json
import os

from lab_helpers import LAST_MONTH, OPENROUTER_BASE_URL, connect, products, profit
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

client, MODEL = connect()

## Step 1: Write a tool

A tool is a Python function that the model is allowed to ask for.

Ours does one small job that we know the model is bad at: add up the contribution profit for one category in one month. It is the same pandas you wrote in notebook 1.

It returns its result as text (JSON), because text is what a model reads. It also reports `is_complete`, which is false if any product in the category has no value.

In [ ]:
def get_category_profit(month: str, category: str) -> str:
    """Return the contribution profit for one category in one month."""
    rows = profit.merge(products, on="product_id")
    rows = rows[(rows["month"] == month) & (rows["category"] == category)]
    total = round(float(rows["contribution_profit"].sum()), 2)
    is_complete = bool(rows["contribution_profit"].notna().all())
    return json.dumps({"month": month, "category": category, "contribution_profit": total, "is_complete": is_complete})

Nothing about AI yet. We can call it ourselves.

In [ ]:
print(get_category_profit(LAST_MONTH, "Hydration"))
print(get_category_profit(LAST_MONTH, "Camp"))

Hydration matches the answer key from notebook 1. Camp comes back with `is_complete: false`, because of the camp stove. The code knows about the trap, every time, without being reminded.

## Step 2: Tell the model the tool exists

The model can't see our Python. We have to describe the tool to it: its name, what it does, and what inputs it takes. The description is written in a standard format that models understand.

In [ ]:
tool_description = {
    "type": "function",
    "function": {
        "name": "get_category_profit",
        "description": "Return the contribution profit for one category in one month.",
        "parameters": {
            "type": "object",
            "properties": {
                "month": {"type": "string", "description": "The month as YYYY-MM"},
                "category": {"type": "string", "description": "Hydration, Camp, or Apparel"},
            },
            "required": ["month", "category"],
        },
    },
}

That is a lot of typing to describe a two-line idea. Hold that thought.

## Step 3: Ask a question, and offer the tool

This is a normal call, as in notebook 2, with one addition: `tools=[tool_description]`. There is no data in the prompt at all this time.

In [ ]:
messages = [
    {"role": "system", "content": "You are a profit analyst. Use the tool to look up figures. Never guess a number."},
    {"role": "user", "content": f"What was the contribution profit for the Hydration category in {LAST_MONTH}?"},
]

response = client.chat.completions.create(model=MODEL, messages=messages, tools=[tool_description], extra_body={"provider": {"require_parameters": True}})
message = response.choices[0].message

print("The model's text reply:", message.content)
print("The model's tool requests:", message.tool_calls)

Read that output carefully, because this is the key moment.

The model did **not** answer the question. Its text reply is empty (or nearly). What it sent back is a **request**: "please run `get_category_profit` with these arguments".

The model can't run our function. It can only ask. Running it is our job.

> **If the tool requests show `None`**, the model answered without using the tool. Run the cell again to reach a different model.

## Step 4: Run the tool and send the result back

We read the arguments the model asked for, call our function, and add two messages to the conversation: the model's request, and our result (with the role `tool`). Then we call the model again.

In [ ]:
assert message.tool_calls, "The model answered without the tool. Run the cell above again."

request = message.tool_calls[0]
arguments = json.loads(request.function.arguments)
print("The model asked for:", request.function.name, arguments)

result = get_category_profit(**arguments)
print("Our function returned:", result)

messages.append(message)
messages.append({"role": "tool", "tool_call_id": request.id, "content": result})

In [ ]:
response = client.chat.completions.create(model=MODEL, messages=messages, tools=[tool_description], extra_body={"provider": {"require_parameters": True}})

print(response.choices[0].message.content)

Now the model answers, and the number in its answer is the one **our code** calculated. The model did no arithmetic. It understood the question, chose the tool, chose the arguments, and put the result into a sentence.

## What you just did, by hand

```
you:    question
model:  "please run get_category_profit(month=..., category=...)"
you:    run the function, send back the result
model:  the answer, using that result
```

That loop has a name. **An agent is a model, with tools, in a loop.** The loop keeps going (ask, run, reply) until the model has what it needs and gives a final answer.

## Step 5: Why we use LangChain

Count what we wrote by hand for one tool and one round of the loop:

| By hand | With LangChain |
|---|---|
| Write the tool description (15 lines of JSON) | `tool(get_category_profit)` writes it from the function |
| Check whether the model asked for a tool | done for you |
| Read the arguments and call the right function | done for you |
| Add the request and the result to the messages | done for you |
| Call the model again, and repeat until it's finished | done for you |

Our real analyst will have six tools and may need several rounds. Writing that loop by hand is tedious and easy to get wrong. **LangChain** is a library that runs the loop for us. That is the whole reason we use it.

First, the tool. LangChain builds the description from what is already in our function: its name, its inputs, and its docstring.

In [ ]:
category_profit_tool = tool(get_category_profit)

print("Name:", category_profit_tool.name)
print("Description:", category_profit_tool.description)
print("Inputs:", category_profit_tool.args)

Compare that with the `tool_description` we typed out in step 2. It is the same information. This is why a tool's docstring matters: the model reads it to decide when to use the tool.

Next, the model and the agent. `ChatOpenAI` is LangChain's way of describing which model to use. `create_agent` puts the model, the tools, and the system prompt together.

In [ ]:
llm = ChatOpenAI(model=MODEL, base_url=OPENROUTER_BASE_URL, api_key=os.getenv("OPENROUTER_API_KEY"), temperature=0)

agent = create_agent(
    model=llm,
    tools=[category_profit_tool],
    system_prompt="You are a profit analyst. Use the tool to look up figures. Never guess a number.",
)

Now ask the same question. `invoke` runs the whole loop. `pretty_print` shows each message in the conversation.

In [ ]:
question = f"What was the contribution profit for the Hydration category in {LAST_MONTH}?"

result = agent.invoke({"messages": [{"role": "user", "content": question}]})

for message in result["messages"]:
    message.pretty_print()

You should see four messages, and they are the same four steps you did by hand:

1. **Human**: the question
2. **Ai**: a tool call (the request)
3. **Tool**: the result of our function
4. **Ai**: the answer

Now a question that needs the tool twice. We don't change any code. The loop simply runs for longer.

In [ ]:
question = f"Which had more contribution profit in {LAST_MONTH}: Hydration or Apparel? Give both figures."

result = agent.invoke({"messages": [{"role": "user", "content": question}]})

for message in result["messages"]:
    message.pretty_print()

## What we now know

- **A tool** is a Python function the model may ask us to run. The code does the calculating.
- **The model never runs anything.** It asks, we run, we reply.
- **An agent** is a model, with tools, in a loop.
- **LangChain** writes the tool descriptions and runs the loop, so we don't have to.

One tool can answer one kind of question. The leader's question needs more: rankings, changes, data problems.

**Next: notebook 5**, where we give an agent six tools and ask the real question.